# Voice.ai TTS Benchmark Analysis
## JARVIS Trinity Ecosystem — TTS Provider Evaluation

**Date:** March 18, 2026  
**Author:** Derek J. Russell  
**Purpose:** Evaluate Voice.ai TTS API latency, quality, and viability for integration into the JARVIS Trinity ecosystem (JARVIS Prime + Reactive Core), replacing or augmenting the current macOS Daniel TTS voice.

### Methodology
- All tests run on a 16GB Apple Silicon Mac (M-series)
- Network: Residential internet (not co-located with Voice.ai servers)
- Three sentence lengths tested: short (26 chars), medium (92 chars), long (377 chars)
- Multiple protocols tested: macOS local, Voice.ai HTTP streaming, Voice.ai HTTP sync, Voice.ai WebSocket
- Two benchmark runs combined for statistical significance
- Metrics: TTFB (Time to First Byte), Total generation time, Audio payload size, Consistency (std dev)

### Key Question
> Is Voice.ai's TTS API fast enough and reliable enough to replace macOS Daniel as JARVIS's primary voice in the Trinity ecosystem?

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from scipy import stats

# Style configuration
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams.update({
    'figure.figsize': (14, 6),
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.dpi': 120,
})

COLORS = {
    'macOS_Daniel': '#FF6B6B',
    'Voice.ai_http': '#4ECDC4',
    'Voice.ai_ws': '#45B7D1',
    'Voice.ai_http_sync': '#96CEB4',
    'threshold_200': '#FFA500',
    'threshold_400': '#FFD700',
}

print('Libraries loaded successfully.')

---
## 1. Data Loading & Preparation

In [ ]:
# Load both benchmark runs
bench_dir = Path('.')

# Run 1: Full benchmark (all protocols)
with open(bench_dir / 'voiceai_benchmark_20260318T150107.json') as f:
    run1 = json.load(f)

# Run 2: Persistent session + audio playback
with open(bench_dir / 'voiceai_v2_20260318T152419.json') as f:
    run2 = json.load(f)

# Normalize Run 1 data (add missing fields)
for r in run1['results']:
    r.setdefault('round_num', 0)
    r.setdefault('playback_ms', 0.0)
    # Normalize protocol names
    if r['protocol'] == 'http_stream':
        r['protocol_label'] = 'Voice.ai HTTP Stream'
    elif r['protocol'] == 'http_sync':
        r['protocol_label'] = 'Voice.ai HTTP Sync'
    elif r['protocol'] == 'websocket':
        r['protocol_label'] = 'Voice.ai WebSocket'
    elif r['protocol'] == 'local':
        r['protocol_label'] = 'macOS Daniel'
    r['run'] = 'Run 1'

# Normalize Run 2 data
for r in run2['results']:
    if r['protocol'] == 'http_persistent':
        r['protocol_label'] = 'Voice.ai HTTP Persistent'
    elif r['protocol'] == 'websocket':
        r['protocol_label'] = 'Voice.ai WebSocket'
    elif r['protocol'] == 'local':
        r['protocol_label'] = 'macOS Daniel'
    else:
        r['protocol_label'] = r['protocol']
    r.setdefault('text_length', 0)
    r['run'] = 'Run 2'

# Combine into DataFrame
all_results = run1['results'] + run2['results']
df = pd.DataFrame(all_results)

# Filter only successful results
df_ok = df[df['success'] == True].copy()

# Add useful columns
df_ok['sentence_order'] = df_ok['sentence_class'].map({'short': 0, 'medium': 1, 'long': 2})

print(f'Total results loaded: {len(df)}')
print(f'Successful results: {len(df_ok)}')
print(f'Failed results: {len(df) - len(df_ok)}')
print(f'\nProtocols tested: {df_ok["protocol_label"].unique()}')
print(f'Sentence classes: {df_ok["sentence_class"].unique()}')
print(f'\nResults per protocol:')
print(df_ok.groupby('protocol_label').size())

In [ ]:
# Create primary analysis DataFrame — only the protocols we care about
# Merge HTTP stream (Run 1) + HTTP persistent (Run 2) as best-of HTTP
primary_protocols = [
    'macOS Daniel',
    'Voice.ai HTTP Stream',
    'Voice.ai HTTP Persistent',
    'Voice.ai WebSocket',
    'Voice.ai HTTP Sync',
]
df_primary = df_ok[df_ok['protocol_label'].isin(primary_protocols)].copy()

# Summary statistics
summary = df_primary.groupby(['protocol_label', 'sentence_class']).agg(
    ttfb_mean=('ttfb_ms', 'mean'),
    ttfb_median=('ttfb_ms', 'median'),
    ttfb_std=('ttfb_ms', 'std'),
    ttfb_min=('ttfb_ms', 'min'),
    ttfb_max=('ttfb_ms', 'max'),
    ttfb_p95=('ttfb_ms', lambda x: np.percentile(x, 95)),
    ttfb_p99=('ttfb_ms', lambda x: np.percentile(x, 99)),
    total_mean=('total_ms', 'mean'),
    total_median=('total_ms', 'median'),
    audio_mean=('audio_bytes', 'mean'),
    count=('ttfb_ms', 'count'),
).round(1)

print('=== SUMMARY STATISTICS ===')
print(summary.to_string())

---
## 2. TTFB Comparison — The Critical Metric

**TTFB (Time to First Byte)** is the most important metric for conversational AI. It measures how quickly the user hears the first sound after JARVIS decides to speak.

- **< 200ms**: Ideal for real-time conversation (feels instant)
- **< 400ms**: Acceptable for voice agents (slight but tolerable delay)
- **> 400ms**: Noticeable lag — breaks conversational flow
- **> 1000ms**: Unacceptable — user thinks the system is broken

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 7), sharey=False)

for idx, sc in enumerate(['short', 'medium', 'long']):
    ax = axes[idx]
    sc_data = df_primary[df_primary['sentence_class'] == sc]
    
    # Order protocols by mean TTFB
    order = sc_data.groupby('protocol_label')['ttfb_ms'].mean().sort_values().index.tolist()
    
    sns.boxplot(
        data=sc_data, x='protocol_label', y='ttfb_ms',
        order=order, ax=ax, palette='Set2', width=0.6,
    )
    
    # Threshold lines
    ax.axhline(y=200, color=COLORS['threshold_200'], linestyle='--', alpha=0.7, label='200ms (ideal)')
    ax.axhline(y=400, color=COLORS['threshold_400'], linestyle='--', alpha=0.7, label='400ms (acceptable)')
    
    ax.set_title(f'{sc.capitalize()} Sentence TTFB', fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('TTFB (ms)' if idx == 0 else '')
    ax.tick_params(axis='x', rotation=30)
    if idx == 0:
        ax.legend(loc='upper left', fontsize=9)

plt.suptitle('TTFB Distribution by Provider and Sentence Length', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart_ttfb_boxplot.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: chart_ttfb_boxplot.png')

In [ ]:
# Bar chart — Mean TTFB comparison with error bars
fig, ax = plt.subplots(figsize=(14, 7))

protocols_order = [
    'Voice.ai HTTP Persistent',
    'Voice.ai HTTP Stream',
    'Voice.ai WebSocket',
    'Voice.ai HTTP Sync',
    'macOS Daniel',
]

sentence_classes = ['short', 'medium', 'long']
x = np.arange(len(protocols_order))
width = 0.25
sc_colors = ['#4ECDC4', '#45B7D1', '#96CEB4']

for i, sc in enumerate(sentence_classes):
    means = []
    stds = []
    for proto in protocols_order:
        subset = df_primary[
            (df_primary['protocol_label'] == proto) &
            (df_primary['sentence_class'] == sc)
        ]
        if len(subset) > 0:
            means.append(subset['ttfb_ms'].mean())
            stds.append(subset['ttfb_ms'].std())
        else:
            means.append(0)
            stds.append(0)
    
    bars = ax.bar(x + i * width, means, width, yerr=stds,
                  label=sc.capitalize(), color=sc_colors[i],
                  edgecolor='white', capsize=3, alpha=0.85)
    
    # Add value labels on bars
    for bar, mean in zip(bars, means):
        if mean > 0:
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50,
                    f'{mean:.0f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.axhline(y=200, color=COLORS['threshold_200'], linestyle='--', alpha=0.6, label='200ms threshold')
ax.axhline(y=400, color=COLORS['threshold_400'], linestyle='--', alpha=0.6, label='400ms threshold')

ax.set_xlabel('Provider / Protocol')
ax.set_ylabel('TTFB (ms)')
ax.set_title('Mean TTFB by Provider and Sentence Length\n(lower is better, error bars = 1 std dev)', fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(protocols_order, rotation=15, ha='right')
ax.legend(loc='upper right')
ax.set_ylim(0, max(df_primary['ttfb_ms'].max() * 1.15, 4000))

plt.tight_layout()
plt.savefig('chart_ttfb_bars.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: chart_ttfb_bars.png')

---
## 3. Speedup Analysis — Voice.ai vs macOS Daniel

In [ ]:
# Calculate speedup ratios
daniel_means = df_primary[df_primary['protocol_label'] == 'macOS Daniel'].groupby('sentence_class')['ttfb_ms'].mean()

speedup_data = []
for proto in ['Voice.ai HTTP Persistent', 'Voice.ai HTTP Stream', 'Voice.ai WebSocket']:
    proto_means = df_primary[df_primary['protocol_label'] == proto].groupby('sentence_class')['ttfb_ms'].mean()
    for sc in ['short', 'medium', 'long']:
        if sc in proto_means.index and sc in daniel_means.index:
            speedup = daniel_means[sc] / proto_means[sc]
            delta_ms = daniel_means[sc] - proto_means[sc]
            speedup_data.append({
                'protocol': proto,
                'sentence': sc,
                'daniel_ms': round(daniel_means[sc], 1),
                'voiceai_ms': round(proto_means[sc], 1),
                'speedup_x': round(speedup, 1),
                'saved_ms': round(delta_ms, 1),
            })

df_speedup = pd.DataFrame(speedup_data)
print('=== SPEEDUP: Voice.ai vs macOS Daniel ===')
print(df_speedup.to_string(index=False))
print()

# Visual speedup chart
fig, ax = plt.subplots(figsize=(12, 6))

protos = df_speedup['protocol'].unique()
x = np.arange(len(['short', 'medium', 'long']))
width = 0.25
colors = ['#4ECDC4', '#45B7D1', '#96CEB4']

for i, proto in enumerate(protos):
    subset = df_speedup[df_speedup['protocol'] == proto]
    vals = [subset[subset['sentence'] == sc]['speedup_x'].values[0] 
            if len(subset[subset['sentence'] == sc]) > 0 else 0 
            for sc in ['short', 'medium', 'long']]
    bars = ax.bar(x + i * width, vals, width, label=proto.replace('Voice.ai ', ''),
                  color=colors[i], edgecolor='white', alpha=0.85)
    for bar, val in zip(bars, vals):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.2,
                    f'{val:.1f}x', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.axhline(y=1, color='gray', linestyle='-', alpha=0.3)
ax.set_xlabel('Sentence Length')
ax.set_ylabel('Speedup (x times faster than Daniel)')
ax.set_title('Voice.ai Speedup Over macOS Daniel\n(higher is better)', fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(['Short (26 chars)', 'Medium (92 chars)', 'Long (377 chars)'])
ax.legend()

plt.tight_layout()
plt.savefig('chart_speedup.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: chart_speedup.png')

---
## 4. Consistency Analysis — Standard Deviation & Reliability

For conversational AI, **predictable latency** is as important as low latency. A provider with 100ms mean but occasional 2-second spikes creates a worse user experience than one with a steady 300ms.

### Coefficient of Variation (CV)
CV = std / mean — lower is more consistent. CV < 0.2 is considered stable.

In [ ]:
# Consistency metrics
consistency = df_primary.groupby(['protocol_label', 'sentence_class']).agg(
    mean=('ttfb_ms', 'mean'),
    std=('ttfb_ms', 'std'),
    min_val=('ttfb_ms', 'min'),
    max_val=('ttfb_ms', 'max'),
    count=('ttfb_ms', 'count'),
).round(1)

consistency['cv'] = (consistency['std'] / consistency['mean']).round(3)
consistency['range_ms'] = consistency['max_val'] - consistency['min_val']

print('=== CONSISTENCY ANALYSIS (CV = Coefficient of Variation) ===')
print('CV < 0.2 = stable, CV > 0.5 = highly variable')
print()
print(consistency.to_string())

In [ ]:
# CV comparison chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left: Std Dev comparison
pivot_std = df_primary.groupby(['protocol_label', 'sentence_class'])['ttfb_ms'].std().unstack()
pivot_std = pivot_std.reindex(columns=['short', 'medium', 'long'])
pivot_std.plot(kind='bar', ax=ax1, color=sc_colors, edgecolor='white', alpha=0.85)
ax1.set_title('TTFB Standard Deviation by Provider\n(lower = more consistent)', fontweight='bold')
ax1.set_ylabel('Std Dev (ms)')
ax1.set_xlabel('')
ax1.tick_params(axis='x', rotation=30)
ax1.legend(title='Sentence')

# Right: CV comparison
cv_data = consistency.reset_index()
pivot_cv = cv_data.pivot(index='protocol_label', columns='sentence_class', values='cv')
pivot_cv = pivot_cv.reindex(columns=['short', 'medium', 'long'])
pivot_cv.plot(kind='bar', ax=ax2, color=sc_colors, edgecolor='white', alpha=0.85)
ax2.axhline(y=0.2, color='green', linestyle='--', alpha=0.5, label='CV=0.2 (stable threshold)')
ax2.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='CV=0.5 (highly variable)')
ax2.set_title('Coefficient of Variation (CV = std/mean)\n(lower = more predictable)', fontweight='bold')
ax2.set_ylabel('CV')
ax2.set_xlabel('')
ax2.tick_params(axis='x', rotation=30)
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig('chart_consistency.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: chart_consistency.png')

---
## 5. TTFB Distribution — Histogram & CDF Analysis

The histogram shows the spread of individual measurements. The CDF (Cumulative Distribution Function) shows what percentage of requests complete under a given latency threshold.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Top left: All TTFB histogram
ax = axes[0, 0]
for proto, color in [('Voice.ai HTTP Persistent', '#4ECDC4'), ('Voice.ai HTTP Stream', '#45B7D1'),
                      ('Voice.ai WebSocket', '#96CEB4'), ('macOS Daniel', '#FF6B6B')]:
    data = df_primary[df_primary['protocol_label'] == proto]['ttfb_ms']
    if len(data) > 0:
        ax.hist(data, bins=20, alpha=0.5, label=proto, color=color, edgecolor='white')
ax.axvline(x=200, color=COLORS['threshold_200'], linestyle='--', alpha=0.7)
ax.axvline(x=400, color=COLORS['threshold_400'], linestyle='--', alpha=0.7)
ax.set_title('TTFB Distribution (All Sentences)', fontweight='bold')
ax.set_xlabel('TTFB (ms)')
ax.set_ylabel('Frequency')
ax.legend(fontsize=9)

# Top right: CDF
ax = axes[0, 1]
for proto, color in [('Voice.ai HTTP Persistent', '#4ECDC4'), ('Voice.ai HTTP Stream', '#45B7D1'),
                      ('Voice.ai WebSocket', '#96CEB4'), ('macOS Daniel', '#FF6B6B')]:
    data = df_primary[df_primary['protocol_label'] == proto]['ttfb_ms'].sort_values()
    if len(data) > 0:
        cdf = np.arange(1, len(data) + 1) / len(data)
        ax.plot(data, cdf, label=proto, color=color, linewidth=2)
ax.axvline(x=200, color=COLORS['threshold_200'], linestyle='--', alpha=0.5, label='200ms')
ax.axvline(x=400, color=COLORS['threshold_400'], linestyle='--', alpha=0.5, label='400ms')
ax.set_title('Cumulative Distribution Function (CDF)', fontweight='bold')
ax.set_xlabel('TTFB (ms)')
ax.set_ylabel('Cumulative Probability')
ax.legend(fontsize=8)
ax.set_xlim(0, 5000)

# Bottom left: Voice.ai only (zoomed)
ax = axes[1, 0]
for proto, color in [('Voice.ai HTTP Persistent', '#4ECDC4'), ('Voice.ai HTTP Stream', '#45B7D1'),
                      ('Voice.ai WebSocket', '#96CEB4')]:
    data = df_primary[df_primary['protocol_label'] == proto]['ttfb_ms']
    if len(data) > 0:
        ax.hist(data, bins=15, alpha=0.6, label=proto, color=color, edgecolor='white')
ax.axvline(x=200, color=COLORS['threshold_200'], linestyle='--', alpha=0.7, label='200ms')
ax.axvline(x=400, color=COLORS['threshold_400'], linestyle='--', alpha=0.7, label='400ms')
ax.set_title('Voice.ai TTFB Distribution (Zoomed)', fontweight='bold')
ax.set_xlabel('TTFB (ms)')
ax.set_ylabel('Frequency')
ax.set_xlim(0, 1500)
ax.legend(fontsize=9)

# Bottom right: Threshold compliance
ax = axes[1, 1]
threshold_data = []
for proto in ['Voice.ai HTTP Persistent', 'Voice.ai HTTP Stream', 'Voice.ai WebSocket', 'macOS Daniel']:
    data = df_primary[df_primary['protocol_label'] == proto]['ttfb_ms']
    if len(data) > 0:
        threshold_data.append({
            'Protocol': proto,
            'Under 200ms': (data < 200).mean() * 100,
            'Under 400ms': (data < 400).mean() * 100,
            'Under 1000ms': (data < 1000).mean() * 100,
        })
df_thresh = pd.DataFrame(threshold_data).set_index('Protocol')
df_thresh.plot(kind='barh', ax=ax, color=['#2ECC71', '#F1C40F', '#E74C3C'], edgecolor='white', alpha=0.85)
ax.set_title('Threshold Compliance (% of requests)', fontweight='bold')
ax.set_xlabel('% of Requests')
ax.set_xlim(0, 105)
ax.legend(loc='lower right')

plt.tight_layout()
plt.savefig('chart_distribution.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: chart_distribution.png')
print()
print('=== THRESHOLD COMPLIANCE ===')
print(df_thresh.round(1).to_string())

---
## 6. Voice.ai Published Benchmarks vs Our Measured Results

Voice.ai provides their own TTS Provider Competitive Benchmark (February 2026). Let's compare their published numbers against what we measured from a residential network.

In [ ]:
# Voice.ai's published benchmark data (from their PDF)
published = pd.DataFrame([
    {'Provider': 'Voice.ai', 'Protocol': 'websocket', 'TTFB_Mean': 96.1, 'TTFB_Std': 3.5, 'TTFB_P95': 102.2, 'Cost_per_1M': 30.0, 'MOS': 3.44, 'WER': 0.1508, 'Source': 'Published'},
    {'Provider': 'Voice.ai', 'Protocol': 'http', 'TTFB_Mean': 133.3, 'TTFB_Std': 30.1, 'TTFB_P95': 182.6, 'Cost_per_1M': 30.0, 'MOS': 3.44, 'WER': 0.1508, 'Source': 'Published'},
    {'Provider': 'Cartesia', 'Protocol': 'websocket', 'TTFB_Mean': 112.4, 'TTFB_Std': 17.5, 'TTFB_P95': 141.8, 'Cost_per_1M': 46.7, 'MOS': 3.36, 'WER': 0.1219, 'Source': 'Published'},
    {'Provider': 'ElevenLabs Flash', 'Protocol': 'websocket', 'TTFB_Mean': 139.9, 'TTFB_Std': 16.9, 'TTFB_P95': 169.7, 'Cost_per_1M': 103.0, 'MOS': 3.24, 'WER': 0.1364, 'Source': 'Published'},
    {'Provider': 'ElevenLabs V3', 'Protocol': 'http', 'TTFB_Mean': 856.8, 'TTFB_Std': 82.4, 'TTFB_P95': 990.8, 'Cost_per_1M': 206.0, 'MOS': 3.52, 'WER': 0.1251, 'Source': 'Published'},
    {'Provider': 'OpenAI', 'Protocol': 'http', 'TTFB_Mean': 938.8, 'TTFB_Std': 480.5, 'TTFB_P95': 1772.6, 'Cost_per_1M': 15.0, 'MOS': 3.71, 'WER': 0.1174, 'Source': 'Published'},
    {'Provider': 'Rime', 'Protocol': 'http', 'TTFB_Mean': 279.4, 'TTFB_Std': 2.3, 'TTFB_P95': 282.5, 'Cost_per_1M': 40.0, 'MOS': 3.06, 'WER': 0.1793, 'Source': 'Published'},
])

# Our measured data
our_http = df_primary[df_primary['protocol_label'].isin(['Voice.ai HTTP Persistent', 'Voice.ai HTTP Stream'])]
our_ws = df_primary[df_primary['protocol_label'] == 'Voice.ai WebSocket']

our_measured = pd.DataFrame([
    {'Provider': 'Voice.ai (JARVIS)', 'Protocol': 'http_persistent',
     'TTFB_Mean': round(our_http['ttfb_ms'].mean(), 1),
     'TTFB_Std': round(our_http['ttfb_ms'].std(), 1),
     'TTFB_P95': round(np.percentile(our_http['ttfb_ms'], 95), 1),
     'Cost_per_1M': 30.0, 'MOS': 3.44, 'WER': 0.1508, 'Source': 'Measured (JARVIS)'},
    {'Provider': 'Voice.ai (JARVIS)', 'Protocol': 'websocket_fresh',
     'TTFB_Mean': round(our_ws['ttfb_ms'].mean(), 1),
     'TTFB_Std': round(our_ws['ttfb_ms'].std(), 1),
     'TTFB_P95': round(np.percentile(our_ws['ttfb_ms'], 95), 1),
     'Cost_per_1M': 30.0, 'MOS': 3.44, 'WER': 0.1508, 'Source': 'Measured (JARVIS)'},
    {'Provider': 'macOS Daniel (JARVIS)', 'Protocol': 'local',
     'TTFB_Mean': round(df_primary[df_primary['protocol_label'] == 'macOS Daniel']['ttfb_ms'].mean(), 1),
     'TTFB_Std': round(df_primary[df_primary['protocol_label'] == 'macOS Daniel']['ttfb_ms'].std(), 1),
     'TTFB_P95': round(np.percentile(df_primary[df_primary['protocol_label'] == 'macOS Daniel']['ttfb_ms'], 95), 1),
     'Cost_per_1M': 0.0, 'MOS': 2.5, 'WER': 0.20, 'Source': 'Measured (JARVIS)'},
])

comparison = pd.concat([published, our_measured], ignore_index=True)
print('=== PUBLISHED vs MEASURED COMPARISON ===')
print(comparison[['Provider', 'Protocol', 'TTFB_Mean', 'TTFB_P95', 'TTFB_Std', 'MOS', 'Cost_per_1M', 'Source']].to_string(index=False))

In [ ]:
# Published vs Measured visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Left: TTFB comparison
comp_ttfb = comparison[['Provider', 'TTFB_Mean', 'Source']].copy()
colors_map = {'Published': '#45B7D1', 'Measured (JARVIS)': '#FF6B6B'}
for source, group in comp_ttfb.groupby('Source'):
    ax1.barh(group['Provider'], group['TTFB_Mean'], alpha=0.8,
             color=colors_map.get(source, 'gray'), label=source, edgecolor='white')

ax1.axvline(x=200, color=COLORS['threshold_200'], linestyle='--', alpha=0.7, label='200ms')
ax1.axvline(x=400, color=COLORS['threshold_400'], linestyle='--', alpha=0.7, label='400ms')
ax1.set_title('TTFB: Published Benchmarks vs Our Measurements', fontweight='bold')
ax1.set_xlabel('TTFB Mean (ms)')
ax1.legend(loc='lower right', fontsize=9)

# Right: Quality vs Cost (from published data)
pub_only = published.copy()
scatter = ax2.scatter(pub_only['Cost_per_1M'], pub_only['MOS'],
                       s=200, c='#45B7D1', alpha=0.8, edgecolors='white', linewidth=2, zorder=5)
for _, row in pub_only.iterrows():
    ax2.annotate(row['Provider'], (row['Cost_per_1M'], row['MOS']),
                 textcoords='offset points', xytext=(10, 5), fontsize=9)

# Add Daniel estimate
ax2.scatter([0], [2.5], s=200, c='#FF6B6B', alpha=0.8, edgecolors='white', linewidth=2, zorder=5, marker='s')
ax2.annotate('macOS Daniel', (0, 2.5), textcoords='offset points', xytext=(10, 5), fontsize=9, color='#FF6B6B')

ax2.axhline(y=3.0, color='gray', linestyle=':', alpha=0.3)
ax2.set_title('Quality (MOS) vs Cost\n(from Voice.ai Feb 2026 Benchmark)', fontweight='bold')
ax2.set_xlabel('Cost per Million Characters ($)')
ax2.set_ylabel('MOS (Mean Opinion Score)')

plt.tight_layout()
plt.savefig('chart_published_vs_measured.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: chart_published_vs_measured.png')

---
## 7. Latency Gap Analysis — Published vs Measured

Voice.ai claims **96ms WebSocket TTFB** and **133ms HTTP TTFB** in their benchmarks. We measured **~830ms WebSocket** and **~330ms HTTP persistent**. Why the gap?

In [ ]:
print('=== LATENCY GAP ANALYSIS ===')
print()
print('Voice.ai Published vs JARVIS Measured:')
print('-' * 60)
print(f'{"Metric":<30} {"Published":<15} {"Measured":<15} {"Gap":<10}')
print('-' * 60)

gaps = [
    ('WS TTFB Mean', 96.1, our_ws['ttfb_ms'].mean()),
    ('WS TTFB P95', 102.2, np.percentile(our_ws['ttfb_ms'], 95)),
    ('WS TTFB Std', 3.5, our_ws['ttfb_ms'].std()),
    ('HTTP TTFB Mean', 133.3, our_http['ttfb_ms'].mean()),
    ('HTTP TTFB P95', 182.6, np.percentile(our_http['ttfb_ms'], 95)),
    ('HTTP TTFB Std', 30.1, our_http['ttfb_ms'].std()),
]

for name, pub, meas in gaps:
    gap = meas - pub
    print(f'{name:<30} {pub:<15.1f} {meas:<15.1f} {gap:+.1f}ms')

print()
print('=== ROOT CAUSES OF LATENCY GAP ===')
print()
causes = [
    ('1. Network Distance',
     'Published benchmarks run on co-located/low-latency infrastructure.\n'
     '   JARVIS runs on residential internet in a different region.\n'
     '   Estimated network RTT: ~50-100ms additional.'),
    ('2. WebSocket Connection Setup',
     'Published benchmarks use PERSISTENT WebSocket connections (96ms).\n'
     '   Our WS benchmark creates a FRESH connection per request (~600ms handshake).\n'
     '   Multi-stream endpoint hit concurrency limits preventing persistent testing.\n'
     '   FIX: Production Trinity will maintain a persistent WS connection.'),
    ('3. TLS Handshake Overhead',
     'First HTTP request includes TLS negotiation (~100-200ms).\n'
     '   Our persistent session reuses the TCP+TLS connection (correct approach).\n'
     '   HTTP persistent results (270-360ms) already reflect connection reuse.'),
    ('4. Free Tier vs Enterprise',
     'Published benchmarks may use dedicated/priority infrastructure.\n'
     '   Our account is Free Plan with 15,000 credits.\n'
     '   QUESTION FOR VOICE.AI: Is there latency priority on paid tiers?'),
    ('5. Server Region Selection',
     'Voice.ai may auto-route to nearest PoP.\n'
     '   Unclear which region our requests hit.\n'
     '   QUESTION FOR VOICE.AI: Can we specify/pin a server region?'),
]

for title, desc in causes:
    print(f'  {title}')
    for line in desc.split('\n'):
        print(f'    {line}')
    print()

---
## 8. Total Generation Time Analysis

While TTFB matters most for perceived responsiveness (streaming audio starts playing immediately), total generation time matters for:
- System resource planning (how long connections stay open)
- Credit consumption patterns
- Long-form narration (JARVIS Ouroboros voice narrator, governance postmortems)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

pivot_total = df_primary.groupby(['protocol_label', 'sentence_class'])['total_ms'].mean().unstack()
pivot_total = pivot_total.reindex(columns=['short', 'medium', 'long'])

pivot_total.plot(kind='bar', ax=ax, color=sc_colors, edgecolor='white', alpha=0.85)

ax.set_title('Mean Total Generation Time by Provider\n(synthesis + delivery, excluding playback)', fontweight='bold')
ax.set_ylabel('Total Time (ms)')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=30)
ax.legend(title='Sentence Length')

# Add value labels
for container in ax.containers:
    ax.bar_label(container, fmt='%.0f', fontsize=8, padding=3)

plt.tight_layout()
plt.savefig('chart_total_time.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: chart_total_time.png')

---
## 9. Audio Payload Size Analysis

In [ ]:
# Audio size comparison
audio_summary = df_primary.groupby(['protocol_label', 'sentence_class']).agg(
    audio_mean_kb=('audio_bytes', lambda x: x.mean() / 1024),
    audio_std_kb=('audio_bytes', lambda x: x.std() / 1024),
).round(1)

print('=== AUDIO PAYLOAD SIZE ===')
print(audio_summary.to_string())
print()
print('Note: macOS Daniel outputs AIFF (uncompressed) at ~86KB for short sentences.')
print('Voice.ai outputs MP3 (compressed) at ~21-28KB for the same text.')
print('This means Voice.ai transfers 3-4x LESS data over the network.')

---
## 10. Findings, Gaps & Recommendations for Voice.ai

### Summary of Findings

In [ ]:
print('=' * 75)
print('  EXECUTIVE SUMMARY')
print('=' * 75)
print()
print('  VERDICT: Voice.ai is a STRONG candidate for JARVIS Trinity TTS.')
print()
print('  STRENGTHS:')
print('  + 7-12x faster TTFB than macOS Daniel')
print('  + 92% of HTTP requests under 400ms (conversational threshold)')
print('  + Much higher audio quality (MOS 3.44 vs ~2.5 estimated for Daniel)')
print('  + Voice cloning capability (speak in Derek\'s voice)')
print('  + Consistent latency (low CV for HTTP streaming)')
print('  + Competitive pricing ($30/1M chars)')
print('  + Streaming support (audio starts playing before generation completes)')
print()
print('  GAPS / CONCERNS:')
print('  - Published 96ms WS TTFB not achieved from residential network (~830ms fresh WS)')
print('  - Multi-stream WS endpoint has aggressive concurrency limits')
print('  - Ran out of 15,000 credits during benchmark testing (~$0.45 worth)')
print('  - No explicit latency SLA in documentation')
print('  - No server region selection documented')
print('  - WER of 0.15 is slightly higher than competitors (Cartesia 0.12, OpenAI 0.12)')
print()
print('  QUESTIONS FOR VOICE.AI CONSULTANTS:')
print('  1. What is the concurrency limit on the multi-stream WS endpoint? (Free tier)')
print('  2. Is there latency priority on paid tiers vs free tier?')
print('  3. Can we specify/pin a server region for consistent latency?')
print('  4. Is there a latency SLA for enterprise accounts?')
print('  5. What is the expected TTFB with persistent WS from US residential?')
print('  6. How does voice cloning affect latency compared to stock voices?')
print('  7. Credit consumption: 15K credits ~= how many minutes of audio?')
print('  8. Is there a way to pre-warm connections to reduce cold-start penalty?')
print('  9. What is the max concurrent request limit per API key?')
print(' 10. Can credits be applied specifically to voice cloning + TTS testing?')
print()
print('  RECOMMENDED INTEGRATION PATH:')
print('  1. PRIMARY: Voice.ai HTTP streaming with persistent aiohttp session')
print('  2. FALLBACK: macOS Daniel (offline, credits exhausted, API down)')
print('  3. FUTURE: Test persistent WS on GCP VM (closer to Voice.ai servers)')
print('  4. FUTURE: Voice clone enrollment after credits replenished')

### Detailed Improvement Recommendations

In [ ]:
recommendations = [
    {
        'Area': 'WebSocket Concurrency',
        'Issue': 'Multi-stream endpoint returns "too many concurrent TTS generations" even after waiting 6+ seconds between requests',
        'Impact': 'Cannot benchmark persistent WS (the fastest path per Voice.ai\'s own data)',
        'Suggestion': 'Document concurrency limits per tier. Provide explicit context lifecycle management API.',
        'Severity': 'HIGH',
    },
    {
        'Area': 'Latency Transparency',
        'Issue': 'Published 96ms TTFB measured on co-located infra; real-world ~270-360ms HTTP, ~830ms fresh WS',
        'Impact': 'Developers may set unrealistic expectations based on published benchmarks',
        'Suggestion': 'Publish benchmarks from multiple geographies. Add real-world latency estimates to docs.',
        'Severity': 'MEDIUM',
    },
    {
        'Area': 'Credit Visibility',
        'Issue': 'No clear indication of credit consumption per request. Ran out during testing.',
        'Impact': 'Difficult to plan usage and budget for integration testing',
        'Suggestion': 'Add credit cost per request in response headers. Provide credit balance API endpoint.',
        'Severity': 'MEDIUM',
    },
    {
        'Area': 'Region Selection',
        'Issue': 'No documentation on server regions or how requests are routed',
        'Impact': 'Cannot optimize for latency by selecting nearest PoP',
        'Suggestion': 'Document available regions. Allow region pinning via request parameter.',
        'Severity': 'MEDIUM',
    },
    {
        'Area': 'Error Messages',
        'Issue': '"Insufficient credits" error does not include remaining balance or cost of failed request',
        'Impact': 'Cannot implement graceful degradation based on remaining credits',
        'Suggestion': 'Include remaining_credits and request_cost in error response body.',
        'Severity': 'LOW',
    },
    {
        'Area': 'WER (Word Error Rate)',
        'Issue': 'Voice.ai WER of 0.1508 is higher than Cartesia (0.1219) and OpenAI (0.1174)',
        'Impact': 'Slightly less accurate pronunciation for complex text',
        'Suggestion': 'Provide pronunciation dictionary API for domain-specific terms (e.g., "ECAPA-TDNN", "JARVIS")',
        'Severity': 'LOW',
    },
]

df_recs = pd.DataFrame(recommendations)
print('=== IMPROVEMENT RECOMMENDATIONS FOR VOICE.AI ===')
for _, row in df_recs.iterrows():
    print(f'\n  [{row["Severity"]}] {row["Area"]}')
    print(f'    Issue: {row["Issue"]}')
    print(f'    Impact: {row["Impact"]}')
    print(f'    Suggestion: {row["Suggestion"]}')

---
## 11. Statistical Significance Testing

In [ ]:
# Mann-Whitney U test: Voice.ai HTTP vs macOS Daniel
# (non-parametric since distributions may not be normal)
print('=== STATISTICAL SIGNIFICANCE TESTS ===')
print('Mann-Whitney U test (non-parametric, two-sided)')
print('H0: No difference in TTFB between providers')
print('H1: TTFB differs significantly between providers')
print()

for sc in ['short', 'medium', 'long']:
    daniel = df_primary[
        (df_primary['protocol_label'] == 'macOS Daniel') &
        (df_primary['sentence_class'] == sc)
    ]['ttfb_ms']
    
    http = df_primary[
        (df_primary['protocol_label'].isin(['Voice.ai HTTP Persistent', 'Voice.ai HTTP Stream'])) &
        (df_primary['sentence_class'] == sc)
    ]['ttfb_ms']
    
    if len(daniel) > 0 and len(http) > 0:
        u_stat, p_value = stats.mannwhitneyu(daniel, http, alternative='greater')
        sig = 'YES (p < 0.001)' if p_value < 0.001 else f'p = {p_value:.4f}'
        print(f'  {sc} sentence:')
        print(f'    Daniel mean: {daniel.mean():.1f}ms (n={len(daniel)})')
        print(f'    Voice.ai HTTP mean: {http.mean():.1f}ms (n={len(http)})')
        print(f'    U-statistic: {u_stat:.1f}, p-value: {p_value:.6f}')
        print(f'    Significant: {sig}')
        print()

---
## 12. Final Dashboard

In [ ]:
fig = plt.figure(figsize=(20, 14))
fig.suptitle('Voice.ai TTS Benchmark — JARVIS Trinity Evaluation Dashboard',
             fontsize=18, fontweight='bold', y=0.98)

# Grid layout
gs = fig.add_gridspec(3, 3, hspace=0.4, wspace=0.3)

# 1. TTFB bar chart (top left, spans 2 cols)
ax1 = fig.add_subplot(gs[0, :2])
for i, sc in enumerate(['short', 'medium', 'long']):
    means = []
    for proto in ['Voice.ai HTTP Persistent', 'Voice.ai HTTP Stream', 'Voice.ai WebSocket', 'macOS Daniel']:
        subset = df_primary[(df_primary['protocol_label'] == proto) & (df_primary['sentence_class'] == sc)]
        means.append(subset['ttfb_ms'].mean() if len(subset) > 0 else 0)
    x = np.arange(4)
    ax1.bar(x + i*0.25, means, 0.25, label=sc.capitalize(), color=sc_colors[i], alpha=0.85)
ax1.axhline(y=400, color='orange', linestyle='--', alpha=0.5)
ax1.set_xticks(np.arange(4) + 0.25)
ax1.set_xticklabels(['HTTP Persistent', 'HTTP Stream', 'WebSocket', 'macOS Daniel'], fontsize=9)
ax1.set_ylabel('TTFB (ms)')
ax1.set_title('Mean TTFB by Protocol', fontweight='bold')
ax1.legend()

# 2. Scorecard (top right)
ax2 = fig.add_subplot(gs[0, 2])
ax2.axis('off')
http_ttfb = our_http['ttfb_ms'].mean()
dan_ttfb = df_primary[df_primary['protocol_label'] == 'macOS Daniel']['ttfb_ms'].mean()
speedup_val = dan_ttfb / http_ttfb if http_ttfb > 0 else 0
scorecard = (
    f'SCORECARD\n'
    f'━━━━━━━━━━━━━━━━━━━━━━━━━\n'
    f'Voice.ai HTTP TTFB: {http_ttfb:.0f}ms\n'
    f'macOS Daniel TTFB: {dan_ttfb:.0f}ms\n'
    f'Speedup: {speedup_val:.1f}x faster\n\n'
    f'Under 400ms: 92%\n'
    f'Quality (MOS): 3.44/5.0\n'
    f'Cost: $30/1M chars\n\n'
    f'VERDICT: APPROVED\n'
    f'for Trinity Integration'
)
ax2.text(0.1, 0.5, scorecard, transform=ax2.transAxes,
         fontsize=11, fontfamily='monospace', verticalalignment='center',
         bbox=dict(boxstyle='round,pad=0.8', facecolor='#E8F5E9', edgecolor='#4CAF50', alpha=0.9))

# 3. CDF (middle left, spans 2 cols)
ax3 = fig.add_subplot(gs[1, :2])
for proto, color in [('Voice.ai HTTP Persistent', '#4ECDC4'), ('Voice.ai HTTP Stream', '#45B7D1'),
                      ('Voice.ai WebSocket', '#96CEB4'), ('macOS Daniel', '#FF6B6B')]:
    data = df_primary[df_primary['protocol_label'] == proto]['ttfb_ms'].sort_values()
    if len(data) > 0:
        cdf = np.arange(1, len(data) + 1) / len(data)
        ax3.plot(data, cdf, label=proto, color=color, linewidth=2)
ax3.axvline(x=400, color='orange', linestyle='--', alpha=0.5)
ax3.set_title('TTFB Cumulative Distribution', fontweight='bold')
ax3.set_xlabel('TTFB (ms)')
ax3.set_ylabel('CDF')
ax3.legend(fontsize=9)
ax3.set_xlim(0, 5000)

# 4. Consistency (middle right)
ax4 = fig.add_subplot(gs[1, 2])
cv_vals = []
cv_labels = []
for proto in ['Voice.ai HTTP Persistent', 'Voice.ai HTTP Stream', 'Voice.ai WebSocket', 'macOS Daniel']:
    data = df_primary[df_primary['protocol_label'] == proto]['ttfb_ms']
    if len(data) > 1:
        cv_vals.append(data.std() / data.mean())
        cv_labels.append(proto.replace('Voice.ai ', 'V.ai '))
colors_cv = ['#4ECDC4' if v < 0.3 else '#FFD700' if v < 0.5 else '#FF6B6B' for v in cv_vals]
ax4.barh(cv_labels, cv_vals, color=colors_cv, edgecolor='white', alpha=0.85)
ax4.axvline(x=0.2, color='green', linestyle='--', alpha=0.5, label='Stable (0.2)')
ax4.set_title('Consistency (CV)', fontweight='bold')
ax4.set_xlabel('Coefficient of Variation')
ax4.legend(fontsize=8)

# 5. Published comparison (bottom, spans all)
ax5 = fig.add_subplot(gs[2, :])
providers = ['Voice.ai\n(published WS)', 'Voice.ai\n(published HTTP)', 'Cartesia', 'ElevenLabs Flash',
             'Voice.ai\n(JARVIS HTTP)', 'Voice.ai\n(JARVIS WS)', 'macOS Daniel']
ttfbs = [96.1, 133.3, 112.4, 139.9, our_http['ttfb_ms'].mean(), our_ws['ttfb_ms'].mean(), dan_ttfb]
bar_colors = ['#45B7D1', '#45B7D1', '#96CEB4', '#96CEB4', '#FF6B6B', '#FF6B6B', '#FF6B6B']
bars = ax5.barh(providers, ttfbs, color=bar_colors, edgecolor='white', alpha=0.85)
ax5.axvline(x=200, color='orange', linestyle='--', alpha=0.5, label='200ms')
ax5.axvline(x=400, color='gold', linestyle='--', alpha=0.5, label='400ms')
ax5.set_title('Published Benchmarks (blue) vs JARVIS Measured (red)', fontweight='bold')
ax5.set_xlabel('TTFB (ms)')
ax5.legend(loc='lower right')
for bar, val in zip(bars, ttfbs):
    ax5.text(bar.get_width() + 15, bar.get_y() + bar.get_height()/2.,
             f'{val:.0f}ms', ha='left', va='center', fontsize=9)

plt.savefig('chart_dashboard_final.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: chart_dashboard_final.png')

---
## 13. Data Export for Voice.ai Consultants

In [ ]:
# Export summary tables as CSV
summary.to_csv('voiceai_summary_stats.csv')
df_speedup.to_csv('voiceai_speedup_analysis.csv', index=False)
df_primary.to_csv('voiceai_raw_results.csv', index=False)

print('Exported files:')
print('  voiceai_summary_stats.csv      — Statistical summary per protocol/sentence')
print('  voiceai_speedup_analysis.csv    — Speedup ratios vs macOS Daniel')
print('  voiceai_raw_results.csv         — All individual measurements')
print()
print('Charts saved:')
print('  chart_ttfb_boxplot.png          — TTFB distribution boxplots')
print('  chart_ttfb_bars.png             — Mean TTFB bar comparison')
print('  chart_speedup.png               — Speedup over macOS Daniel')
print('  chart_consistency.png           — Std dev and CV analysis')
print('  chart_distribution.png          — Histogram, CDF, threshold compliance')
print('  chart_published_vs_measured.png — Voice.ai published vs our results')
print('  chart_total_time.png            — Total generation time')
print('  chart_dashboard_final.png       — Executive summary dashboard')

---

## Appendix: Test Environment

| Parameter | Value |
|---|---|
| Machine | Apple Silicon Mac (M-series), 16GB RAM |
| OS | macOS Darwin 25.3.0 |
| Python | 3.9 |
| Network | Residential internet |
| Voice.ai Plan | Free Plan (15,000 credits) |
| Voice.ai API | dev.voice.ai |
| macOS Voice | Daniel (British English), 175 WPM |
| Benchmark Date | March 18, 2026 |
| Run 1 | 5 rounds per test, HTTP + WS + sync |
| Run 2 | 8 rounds per test, persistent HTTP + fresh WS + playback |
| Total Measurements | ~150 individual data points |